# Imports

In [ ]:
!pip install torch_geometric

In [2]:
import os
import numpy as np
import pandas as pd
import copy
import pprint
import gc
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, LayerNorm as GraphLayerNorm
from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader

import joblib
from datetime import datetime, timezone, timedelta

from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, confusion_matrix, precision_recall_curve
from scipy.special import expit # It is the optimized sigmoid of Scipy


In [3]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


# Experiment Manager

In [4]:
class ExperimentManager:
    def __init__(self, log_file="./logs/experiments_log_gnn.csv", model_dir="./saved_models"):
        self.log_file = log_file
        self.model_dir = model_dir
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

    def log_experiment(self, model_config=None, model_name=None, params=None, metrics=None, model_object=None):
        """
        Record an experiment in CSV format and optionally save the model.

        model_config: Recommended dict:
        - model_name (str)
        - type (str)
        - model_params (dict) # ONLY model hyperparameters
        - prob_threshold (float) # Decision threshold for probabilities (for metrics computation)
        - data_params (dict) # Optional
        - extra_params (dict) # Optional

        model_name: Name of the model (str)

        params: (legacy) Extra dict (for compatibility)
        metrics: Dict with results
        model_object: Model object (for saving)
        """

        if metrics is None:
            metrics = {}
        if params is None:
            params = {}

        # Timezone Argentina
        tz = timezone(timedelta(hours=-3))
        now = datetime.now(tz)

        # Input
        if model_config is not None:
            mname = model_config.get("model_name", model_name)
            mtype = model_config.get("type", None)
            model_params = model_config.get("model_params", {})
            threshold = model_config.get("prob_threshold", None)
            data_params = model_config.get("data_params", {})
            extra_params = model_config.get("extra_params", {})
        else:
            # legacy mode
            mname = model_name
            mtype = params.get("type", None)
            threshold = params.get("prob_threshold", None)
            model_params = params
            data_params = {}
            extra_params = {}

        # Create record
        entry = {
            "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
            "model_name": mname,
        }
        if mtype is not None:
            entry["type"] = mtype
        if threshold is not None:
            entry["prob_threshold"] = threshold

        # Prefijos para evitar colisiones de nombres
        entry.update({f"hp_{k}": v for k, v in (model_params or {}).items()})
        entry.update({f"data_{k}": v for k, v in (data_params or {}).items()})
        entry.update({f"extra_{k}": v for k, v in {**extra_params, **params}.items()
                      if k not in ("type", "prob_threshold")})

        entry.update(metrics)

        # Save in csv
        df_new = pd.DataFrame([entry])
        if os.path.exists(self.log_file):
            df_new.to_csv(self.log_file, mode="a", header=False, index=False)
        else:
            df_new.to_csv(self.log_file, mode="w", header=True, index=False)

        print(f"Experiment recorded in {self.log_file}")

        # Save model
        if model_object is not None:
            metric_key = "AUC-PR" if "AUC-PR" in metrics else ("F1" if "F1" in metrics else None)
            metric_val = metrics.get(metric_key, 0) if metric_key else 0
            safe_key = metric_key or "metric"
            filename = f"{mname}_{now.strftime('%Y%m%d_%H%M')}_{safe_key}_{float(metric_val):.4f}"

            if "sklearn" in str(type(model_object)):
                import joblib
                joblib.dump(model_object, os.path.join(self.model_dir, f"{filename}.joblib"))
            else:
                import torch
                torch.save(model_object.state_dict(), os.path.join(self.model_dir, f"{filename}.pth"))

            print(f"Saved model: {filename}")


# Global instance
exp_manager = ExperimentManager()

# Load data

In [5]:
class NF_IDS_Dataset(Dataset):
    def __init__(self, root_dir, split='train'):
        # root_dir: (eg: "./dataset_processed")
        # split: 'train', 'val' or 'test'
        self.split_dir = os.path.join(root_dir, split)
        super().__init__(self.split_dir)

        # List files ordered numerically to respect the time
        self.files = sorted(
            [f for f in os.listdir(self.split_dir) if f.endswith('.pt')],
            key=lambda x: int(x.split('_')[1].split('.')[0])
        )

    def len(self):
        return len(self.files)

    def get(self, idx):
        data = torch.load(
            os.path.join(self.split_dir, self.files[idx]),
            weights_only=False # to allow loading complex graph objects
        )
        return data

# Models

## Static GNN Identity

In [29]:
class StaticGNN_Identity(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2, output_bias_init=None):
        super(StaticGNN_Identity, self).__init__()

        self.hidden_dim = hidden_dim
        self.dropout_rate = dropout

        # IDENTITY PROJECTION
        # Layer to transform the raw edge attributes into a node vector
        self.edge_proj = nn.Linear(edge_dim, node_dim)

        # GNN INPUT DIMENSIONS
        # Since (What the IP sends + What the IP receives) will be concatenated, the input is double.
        gnn_input_dim = 2 * node_dim

        # GNN LAYER 1
        self.gnn1 = GATv2Conv(
            in_channels=gnn_input_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,
            concat=False,
            dropout=dropout
        )
        self.norm1 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # GNN LAYER 2
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False,
            dropout=dropout
        )
        self.norm2 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # EDGE CLASSIFIER
        classifier_input_dim = (2 * hidden_dim) + edge_dim

        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # Bias Init
        if output_bias_init is not None:
            self.classifier[-1].bias.data.fill_(output_bias_init)

    # --- Utils for building "identity" (node features) ---
    def manual_scatter_mean(self, src, index, dim_size):
        out = torch.zeros((dim_size, src.size(1)), device=src.device)
        out.index_add_(0, index, src)
        ones = torch.ones(src.size(0), 1, device=src.device)
        count = torch.zeros(dim_size, 1, device=src.device)
        count.index_add_(0, index, ones)
        count[count < 1] = 1
        return out / count

    def forward(self, x, edge_index, edge_attr):
        # Check empty graph
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # ---------------------------------------------------------
        # STEP 1: BUILDING "IDENTITY" (node features)
        # ---------------------------------------------------------
        edge_embeddings = self.edge_proj(edge_attr)
        edge_embeddings = F.relu(edge_embeddings)

        src, dst = edge_index
        num_nodes = x.size(0)

        # Dual Identity
        x_out = self.manual_scatter_mean(edge_embeddings, src, dim_size=num_nodes)
        x_in  = self.manual_scatter_mean(edge_embeddings, dst, dim_size=num_nodes)

        # Concatenate [What IP sends | What IP recibes]
        x_input = torch.cat([x_out, x_in], dim=1)

        # ---------------------------------------------------------
        # STEP 2: GNN LAYERS
        # ---------------------------------------------------------

        # Layer 1
        h1 = self.gnn1(x_input, edge_index, edge_attr=edge_attr)
        h1 = self.norm1(h1) # LayerNorm
        h1 = F.elu(h1)
        h1 = F.dropout(h1, p=self.dropout_rate, training=self.training)

        # Layer 2
        h2 = self.gnn2(h1, edge_index, edge_attr=edge_attr)
        h2 = self.norm2(h2) # LayerNorm
        h2 = F.elu(h2)

        # ---------------------------------------------------------
        # STEP 3: CLASSIFICATION
        # ---------------------------------------------------------
        src, dst = edge_index
        edge_rep = torch.cat([h2[src], h2[dst], edge_attr], dim=1)

        return self.classifier(edge_rep)

## ST_GNN_Identity

In [18]:
class ST_GNN_Identity(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2, output_bias_init=None):
        super(ST_GNN_Identity, self).__init__()

        self.hidden_dim = hidden_dim
        self.dropout_rate = dropout

        # IDENTITY PROJECTION
        # Layer to transform the raw edge attributes into a node vector
        self.edge_proj = nn.Linear(edge_dim, node_dim)

        # GNN INPUT DIMENSIONS
        # Since (What the IP sends + What the IP receives) will be concatenated, the input is double.
        gnn_input_dim = 2 * node_dim

        # GNN LAYER 1
        self.gnn1 = GATv2Conv(
            in_channels=gnn_input_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,
            concat=False,
            dropout=dropout
        )
        self.norm1 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # GNN LAYER 2
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False,
            dropout=dropout
        )
        self.norm2 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # TEMPORAL (GRU)
        self.gru = nn.GRUCell(input_size=hidden_dim, hidden_size=hidden_dim)

        # EDGE CLASSIFIER
        classifier_input_dim = (2 * hidden_dim) + edge_dim

        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # Bias Init
        if output_bias_init is not None:
            self.classifier[-1].bias.data.fill_(output_bias_init)

        self.node_memory = {}

    # --- UTILS ---
    def manual_scatter_mean(self, src, index, dim_size):
        out = torch.zeros((dim_size, src.size(1)), device=src.device)
        out.index_add_(0, index, src)

        ones = torch.ones(src.size(0), 1, device=src.device)
        count = torch.zeros(dim_size, 1, device=src.device)
        count.index_add_(0, index, ones)

        count[count < 1] = 1
        return out / count

    def get_memory(self, ids, device):
        mem_list = []
        for i in ids:
            i = i.item()
            if i in self.node_memory:
                mem_list.append(self.node_memory[i])
            else:
                mem_list.append(torch.zeros(self.hidden_dim, device=device))
        return torch.stack(mem_list)

    def update_memory(self, ids, h_new):
        h_detached = h_new.detach()
        for idx, global_id in enumerate(ids):
            self.node_memory[global_id.item()] = h_detached[idx]

    def detach_all_memory(self):
        for k in self.node_memory:
            self.node_memory[k] = self.node_memory[k].detach()

    def reset_memory(self):
        self.node_memory = {}

    def forward(self, x, edge_index, edge_attr, global_ids):
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # ---------------------------------------------------------
        # STEP 1: BUILDING "IDENTITY" (node features)
        # ---------------------------------------------------------

        edge_embeddings = self.edge_proj(edge_attr)
        edge_embeddings = F.relu(edge_embeddings) # Activación inicial

        src, dst = edge_index
        num_nodes = x.size(0)

        # Dual Identity
        x_out = self.manual_scatter_mean(edge_embeddings, src, dim_size=num_nodes)
        x_in  = self.manual_scatter_mean(edge_embeddings, dst, dim_size=num_nodes)

        # Concatenate [What IP sends | What IP recibes]
        # Size: [Num_Nodes, 2 * Node_Dim]
        x_input = torch.cat([x_out, x_in], dim=1)

        # ---------------------------------------------------------
        # STEP 2: GNN PROCESSING + MEMORY
        # ---------------------------------------------------------
        h_prev = self.get_memory(global_ids, x.device)

        # GNN LAYER 1
        z = self.gnn1(x_input, edge_index, edge_attr=edge_attr)
        z = self.norm1(z) # LayerNorm
        z = F.elu(z)
        z = F.dropout(z, p=self.dropout_rate, training=self.training)

        # GNN LAYER 2
        z = self.gnn2(z, edge_index, edge_attr=edge_attr)
        z = self.norm2(z)
        z = F.elu(z)

        # GRU LAYER
        h_current = self.gru(z, h_prev)
        self.update_memory(global_ids, h_current)

        # ---------------------------------------------------------
        # STEP 3: CLASSIFICATION
        # ---------------------------------------------------------
        edge_rep = torch.cat([h_current[src], h_current[dst], edge_attr], dim=1)

        return self.classifier(edge_rep)

# Functions

This handles the complex logic: Truncated Backpropagation for ST-GNN and detailed metric calculations (including False Positives).

## Metrics

In [7]:
def calculate_metrics_gnn(y_true, y_pred_logits, prob_threshold=0.5):
    """
    y_true: List or array of actual labels (0 or 1).
    y_pred_logits: List or array of raw model outputs (before sigmoid).
    """
    y_true = np.array(y_true)
    logits = np.array(y_pred_logits)

    # Convert logits to probabilities, and to classes (0 or 1) depend on prob_threshold
    probs = expit(logits)
    preds = (probs > prob_threshold).astype(int)

    # Threshold-dependent metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    f2 = fbeta_score(y_true, preds, beta=2, zero_division=0)

    # Threshold-independent metrics
    try:
        ap = average_precision_score(y_true, probs)
        roc = roc_auc_score(y_true, probs)
    except ValueError:
        # This occurs if y_true has only one class (e.g., only benign in the batch)
        ap = 0.0
        roc = 0.5

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2, "AUC-PR": ap, "AUC-ROC": roc,
        "FPR": fpr, "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn), "Total_Flows": len(y_true)
    }



## Train and eval

In [8]:
def train_epoch(model, loader, optimizer, criterion, device, is_temporal=False, batch_steps=10):
    model.train()
    total_loss = 0
    steps = 0  # Count of valid graphs processed (not empty)

    # Loss accumulator for Truncated Backprop
    batch_loss = 0

    # Iterate over the Loader
    # batch_idx helps to know if we are on the last element
    for batch_idx, data in enumerate(loader):
        data = data.to(device)

        # If it is an empty graph, skip
        if data.x.shape[0] == 0:
            continue

        # Forward
        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        # Loss calculation
        loss = criterion(out.view(-1), data.y)
        batch_loss += loss
        steps += 1

        # Backpropagation:
        # 1. Each 'batch_steps' valid steps (e.g., every 10 graphs)
        # 2. Or if it is in the LAST batch of the loader (so as not to lose the remainder)
        is_last_batch = (batch_idx == len(loader) - 1)

        if (steps > 0 and steps % batch_steps == 0) or is_last_batch:
            # Only update if there's something accumulated
            if batch_loss > 0:
                optimizer.zero_grad()
                batch_loss.backward()
                optimizer.step()

                # Reset
                total_loss += batch_loss.item()
                batch_loss = 0

                # IMPORTANT for ST-GNN: Detach memory
                if is_temporal:
                    model.detach_all_memory()

    # Average loss per valid step
    return total_loss / steps if steps > 0 else 0

@torch.no_grad()
def evaluate(model, loader, criterion, device, prob_threshold, is_temporal=False):
    model.eval()
    all_logits = []
    all_trues = []
    total_loss = 0
    steps = 0

    # Clear the memory before starting the evaluation (only if it's temporal)
    if is_temporal:
        model.node_memory = {}

    # Don't need enumerate here because we're not doing backprop
    for data in loader:
        data = data.to(device)

        if data.x.shape[0] == 0: continue

        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        loss = criterion(out.view(-1), data.y)
        total_loss += loss.item()

        all_logits.extend(out.view(-1).cpu().numpy())
        all_trues.extend(data.y.cpu().numpy())
        steps += 1

    metrics = calculate_metrics_gnn(all_trues, all_logits, prob_threshold)
    metrics["Loss"] = total_loss / steps if steps > 0 else 0
    return metrics

## Optimal threshold

In [9]:
def find_optimal_threshold(model, loader, device, is_temporal=False):
    model.eval()

    if is_temporal:
        # Clear the memory before starting the evaluation (only if it's temporal)
        model.node_memory = {}

    all_probs = []
    all_trues = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            if is_temporal:
                out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
            else:
                out = model(data.x, data.edge_index, data.edge_attr)

            # Convert Logits to Probabilities (Sigmoid)
            probs = torch.sigmoid(out.view(-1))

            all_probs.extend(probs.cpu().numpy())
            all_trues.extend(data.y.cpu().numpy())

    all_trues = np.array(all_trues)
    all_probs = np.array(all_probs)

    # 1. Precision-Recall Curve
    precisions, recalls, thresholds = precision_recall_curve(all_trues, all_probs)

    # 2. Calculate F1 for each possible threshold
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    f1_scores = f1_scores[:-1]

    # 3. Find max
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    best_rec = recalls[best_idx]
    best_prec = precisions[best_idx]

    print(f"\nBEST THRESHOLD FOUND: {best_threshold:.4f}")
    print(f"   F1 Score : {best_f1:.4f}")
    print(f"   Recall   : {best_rec:.4f}")
    print(f"   Precision: {best_prec:.4f}")

    # Probability Diagnosis
    avg_benign = np.mean(all_probs[all_trues == 0])
    avg_attack = np.mean(all_probs[all_trues == 1])
    print(f"\nProbability Diagnosis:")
    print(f"   Average assigned to Benignos: {avg_benign:.4f}")
    print(f"   Average assigned to Attacks : {avg_attack:.4f}")

    return best_threshold

# Auxiliary

In [10]:
ROOT_PATH = "./dataset_processed"

In [11]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)

Train size: 1998 | Val size: 428


### STATIC: Bias init - LayerNorm - Identity (STATIC)

In [23]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
#HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5

search_space = {
    'hidden_dim': [32]#, 64]
}


Using device: cpu


In [30]:
model_config = {
    "model_name": "Static_GNN_biasinit",
    "type": "GNN_static_BiasOn",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": None,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [26]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_gnn_biasinit_robust_identity.csv")

In [31]:
# Load existing history to avoid repeating
existing_experiments = []
if os.path.exists(exp_manager.log_file):
    try:
        df_log = pd.read_csv(exp_manager.log_file)
        if 'model_name' in df_log.columns:
            existing_experiments = df_log['model_name'].values.tolist()
        print(f"History loaded. {len(existing_experiments)} experiments already performed")
    except:
        print("The current log could not be read, we will start from scratch")

count = 0

for h_dim in search_space['hidden_dim']:
    count += 1

    # Name
    exp_id = f"Static_GNN_H{h_dim}_BiasOn_robust_identity"
    print(f"\n{exp_id}")

    if exp_id in existing_experiments:
        print(f" Skipping {exp_id} (Already registered in CSV)")
        continue

    print(f"\n Starting: {exp_id}")

    # Preventive Memory Cleaning (Before loading anything)
    gc.collect()
    torch.cuda.empty_cache()

    # Update model_config
    model_config['model_name'] = exp_id
    model_config["model_params"]["hidden_dim"] = h_dim

    try:
        # Instantiate
        model = StaticGNN_Identity(**model_config['model_params']).to(DEVICE)

        # Configure training
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))

        # Deepcopy vars
        best_aucpr = 0.0
        best_wts = copy.deepcopy(model.state_dict())
        best_metrics = {}

        for epoch in range(EPOCHS):
            loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE, is_temporal=False, batch_steps=BATCH_STEPS)
            metrics = evaluate(model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=False)

            if (epoch+1) % 2 == 0: # Print every 2 epochs
                print(f"   Ep {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val Rec: {metrics['Recall']:.4f}")

            if metrics['AUC-PR'] > best_aucpr:
                best_aucpr = metrics['AUC-PR']
                best_metrics = metrics
                best_wts = copy.deepcopy(model.state_dict())

        # Restore and save
        model.load_state_dict(best_wts)
        exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=model)
        print(f"   Best AUC-PR: {best_aucpr:.4f} (FPR: {metrics['FPR']:.4f})")
        _=find_optimal_threshold(model, val_loader, DEVICE, False)
        print("=============\n")

    except Exception as e:
        print(f"   FATAL ERROR in {exp_id}: {str(e)}")
        continue

    # Clean memory
    del model
    del optimizer
    del criterion
    del best_wts
    gc.collect()
    torch.cuda.empty_cache()




History loaded. 1 experiments already performed

Static_GNN_H32_BiasOn_robust_identity

 Starting: Static_GNN_H32_BiasOn_robust_identity
   Ep 2 | Loss: 0.1910 | Val AUC-PR: 0.1089 | Val Rec: 0.0000
   Ep 4 | Loss: 0.1866 | Val AUC-PR: 0.0994 | Val Rec: 0.0000
   Ep 6 | Loss: 0.1822 | Val AUC-PR: 0.1582 | Val Rec: 0.0000
   Ep 8 | Loss: 0.1822 | Val AUC-PR: 0.1761 | Val Rec: 0.0000
   Ep 10 | Loss: 0.1720 | Val AUC-PR: 0.1844 | Val Rec: 0.0000
   Ep 12 | Loss: 0.1730 | Val AUC-PR: 0.2339 | Val Rec: 0.0000
   Ep 14 | Loss: 0.1673 | Val AUC-PR: 0.2792 | Val Rec: 0.0000
   Ep 16 | Loss: 0.1617 | Val AUC-PR: 0.3304 | Val Rec: 0.0000
   Ep 18 | Loss: 0.1696 | Val AUC-PR: 0.3329 | Val Rec: 0.1350
   Ep 20 | Loss: 0.1597 | Val AUC-PR: 0.3666 | Val Rec: 0.1444
   Ep 22 | Loss: 0.1568 | Val AUC-PR: 0.3877 | Val Rec: 0.1421
   Ep 24 | Loss: 0.1550 | Val AUC-PR: 0.3972 | Val Rec: 0.1505
   Ep 26 | Loss: 0.1527 | Val AUC-PR: 0.3901 | Val Rec: 0.1487
   Ep 28 | Loss: 0.1529 | Val AUC-PR: 0.4281 | V

In [32]:
find_optimal_threshold(model, val_loader, DEVICE, False)


BEST THRESHOLD FOUND: 0.4509
   F1 Score : 0.4628
   Recall   : 0.3743
   Precision: 0.6061

Probability Diagnosis:
   Average assigned to Benignos: 0.0678
   Average assigned to Attacks : 0.4034


np.float32(0.45090413)

### ST: Bias init - LayerNorm - Identity

In [12]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
#HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5

search_space = {
    'hidden_dim': [32]#, 64]
}


Using device: cpu


In [13]:
model_config = {
    "model_name": "ST_GNN_biasinit",
    "type": "GNN_and_GRU_BiasOn",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": None,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [14]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_gnn_biasinit_robust_identity.csv")

In [21]:
# Load existing history to avoid repeating
existing_experiments = []
if os.path.exists(exp_manager.log_file):
    try:
        df_log = pd.read_csv(exp_manager.log_file)
        if 'model_name' in df_log.columns:
            existing_experiments = df_log['model_name'].values.tolist()
        print(f"History loaded. {len(existing_experiments)} experiments already performed")
    except:
        print("The current log could not be read, we will start from scratch")

count = 0

for h_dim in search_space['hidden_dim']:
    count += 1

    # Name
    exp_id = f"ST_GNN_H{h_dim}_BiasOn_robust_identity"
    print(f"\n{exp_id}")

    if exp_id in existing_experiments:
        print(f" Skipping {exp_id} (Already registered in CSV)")
        continue

    print(f"\n Starting: {exp_id}")

    # Preventive Memory Cleaning (Before loading anything)
    gc.collect()
    torch.cuda.empty_cache()

    # Update model_config
    model_config['model_name'] = exp_id
    model_config["model_params"]["hidden_dim"] = h_dim

    try:
        # Instantiate
        model = ST_GNN_Identity(**model_config['model_params']).to(DEVICE)

        # Configure training
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))

        # Deepcopy vars
        best_aucpr = 0.0
        best_wts = copy.deepcopy(model.state_dict())
        best_metrics = {}

        for epoch in range(EPOCHS):
            loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE, is_temporal=True, batch_steps=BATCH_STEPS)
            metrics = evaluate(model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=True)

            if (epoch+1) % 2 == 0: # Print every 2 epochs
                print(f"   Ep {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics['AUC-PR']:.4f} | Val Rec: {metrics['Recall']:.4f}")

            if metrics['AUC-PR'] > best_aucpr:
                best_aucpr = metrics['AUC-PR']
                best_metrics = metrics
                best_wts = copy.deepcopy(model.state_dict())

        # Restore and save
        model.load_state_dict(best_wts)
        exp_manager.log_experiment(model_config=model_config, metrics=best_metrics, model_object=model)
        print(f"   Best AUC-PR: {best_aucpr:.4f} (FPR: {metrics['FPR']:.4f})")
        _=find_optimal_threshold(model, val_loader, DEVICE, True)
        print("=============\n")

    except Exception as e:
        print(f"   FATAL ERROR in {exp_id}: {str(e)}")
        continue

    # Clean memory
    del model
    del optimizer
    del criterion
    del best_wts
    gc.collect()
    torch.cuda.empty_cache()





ST_GNN_H32_BiasOn_robust_identity

 Starting: ST_GNN_H32_BiasOn_robust_identity
   Ep 2 | Loss: 0.1934 | Val AUC-PR: 0.1641 | Val Rec: 0.0000
   Ep 4 | Loss: 0.1497 | Val AUC-PR: 0.5472 | Val Rec: 0.0860
   Ep 6 | Loss: 0.1415 | Val AUC-PR: 0.6521 | Val Rec: 0.2373
   Ep 8 | Loss: 0.1505 | Val AUC-PR: 0.5605 | Val Rec: 0.0225
   Ep 10 | Loss: 0.1007 | Val AUC-PR: 0.6050 | Val Rec: 0.5354
   Ep 12 | Loss: 0.0865 | Val AUC-PR: 0.2568 | Val Rec: 0.5716
   Ep 14 | Loss: 0.0951 | Val AUC-PR: 0.1415 | Val Rec: 0.0163
   Ep 16 | Loss: 0.1020 | Val AUC-PR: 0.1706 | Val Rec: 0.6843
   Ep 18 | Loss: 0.0985 | Val AUC-PR: 0.0939 | Val Rec: 0.4079
   Ep 20 | Loss: 0.0924 | Val AUC-PR: 0.6497 | Val Rec: 0.5348
   Ep 22 | Loss: 0.1256 | Val AUC-PR: 0.5577 | Val Rec: 0.4563
   Ep 24 | Loss: 0.0895 | Val AUC-PR: 0.5484 | Val Rec: 0.7305
   Ep 26 | Loss: 0.0863 | Val AUC-PR: 0.5812 | Val Rec: 0.1606
   Ep 28 | Loss: 0.0869 | Val AUC-PR: 0.6361 | Val Rec: 0.3074
   Ep 30 | Loss: 0.0765 | Val AUC-PR: 0.6

# Fix logs

In [37]:
!ls logs_orig/

experiments_log_gnn_biasinit.csv
experiments_log_gnn_biasinit_robust.csv
experiments_log_gnn_biasinit_robust_identity.csv
experiments_log_mlp_biasinit.csv
experiments_log_rf.csv
experiments_log_rnn_biasinit.csv
experiments_log_rnn_biasinitOLD.csv


In [102]:
!ls saved_models

EdgeGRU_BiasOn_20260202_1235_AUC-PR_0.0581.pth
EdgeGRUrobust_BiasOn_20260202_1500_AUC-PR_0.2874.pth
EdgeMLP_20260128_1620_AUC-PR_0.0847.pth
EdgeMLP_W1_H32_20260128_1706_AUC-PR_0.1642.pth
EdgeMLP_W1_H64_20260128_1714_AUC-PR_0.1575.pth
EdgeMLP_W2_H32_20260128_1721_AUC-PR_0.1510.pth
EdgeMLP_W2_H64_20260128_1729_AUC-PR_0.1556.pth
EdgeMLP_W5_H32_20260128_1736_AUC-PR_0.1531.pth
EdgeMLP_W5_H64_20260128_1744_AUC-PR_0.1554.pth
RandomForest_20260128_1626_AUC-PR_0.2861.joblib
StaticGNN_biasinit_20260128_1901_AUC-PR_0.1628.pth
Static_GNN_H32_BiasOn_robust_identity_20260205_1416_AUC-PR_0.4760.pth
ST_GNN_biasinit_20260128_2017_AUC-PR_0.2240.pth
ST_GNN_H32_BiasOn_20260128_2215_AUC-PR_0.2414.pth
ST_GNN_H32_BiasOn_robust_20260202_1655_AUC-PR_0.2246.pth
ST_GNN_H32_BiasOn_robust_identity_20260205_1302_AUC-PR_0.7269.pth
ST_GNN_H64_BiasOn_20260128_2334_AUC-PR_0.1860.pth


## experiments_log_gnn_biasinit.csv

In [47]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids-mitre/logs_orig/experiments_log_gnn_biasinit.csv")
df



,timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,...,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
0,2026-01-28 19:01:37,StaticGNN_biasinit,GNN_traditional_BiasOn,0.5,16,32,32,0.2,-2.9968,30,...,0.000000,0.162752,0.769404,0.000000,0,0,757206,31329,788535,0.220476
1,2026-01-28 20:17:10,ST_GNN_biasinit,GNN_and_GRU_BiasOn,0.5,16,32,32,0.2,-2.9968,30,...,0.011463,0.223965,0.766816,0.000030,288,23,757183,31041,788535,0.215759
2,2026-01-28 22:15:08,ST_GNN_H32_BiasOn,GNN_and_GRU_BiasOn,0.5,16,32,32,0.2,-2.9968,60,...,0.181967,0.241375,0.784873,0.005213,4882,3947,753259,26447,788535,0.209327
3,2026-01-28 23:34:18,ST_GNN_H64_BiasOn,GNN_and_GRU_BiasOn,0.5,16,32,64,0.2,-2.9968,60,...,0.186363,0.185989,0.771457,0.005673,5018,4296,752910,26311,788535,0.211019


In [48]:
df["model_name_fix"] = ["StaticGNN_biasinit", "ST_GNN_biasinit", "ST_GNN_H32_BiasOn", "ST_GNN_H64_BiasOn"]
df["type_fix"] = ["StaticGNN_biasinit", "ST_GNN_biasinit", "ST_GNN_biasinit", "ST_GNN_biasinit"]

# Get current columns
current_columns = df.columns.tolist()

# Define columns to move
fixed_columns = ['model_name_fix', 'type_fix']
original_columns = ['model_name', 'type']

# Remove these columns from their current position
remaining_columns = [col for col in current_columns if col not in (['timestamp'] + fixed_columns + original_columns)]

# Construct the new order
new_column_order = [
    'timestamp',
    'model_name_fix',
    'type_fix'
] + remaining_columns + original_columns

# Apply the new order to the DataFrame
df = df[new_column_order]
df.to_csv("/content/drive/MyDrive/nids-mitre/logs/experiments_log_gnn_biasinit.csv", index=False)

df

,timestamp,model_name_fix,type_fix,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,...,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss,model_name,type
0,2026-01-28 19:01:37,StaticGNN_biasinit,StaticGNN_biasinit,0.5,16,32,32,0.2,-2.9968,30,...,0.769404,0.000000,0,0,757206,31329,788535,0.220476,StaticGNN_biasinit,GNN_traditional_BiasOn
1,2026-01-28 20:17:10,ST_GNN_biasinit,ST_GNN_biasinit,0.5,16,32,32,0.2,-2.9968,30,...,0.766816,0.000030,288,23,757183,31041,788535,0.215759,ST_GNN_biasinit,GNN_and_GRU_BiasOn
2,2026-01-28 22:15:08,ST_GNN_H32_BiasOn,ST_GNN_biasinit,0.5,16,32,32,0.2,-2.9968,60,...,0.784873,0.005213,4882,3947,753259,26447,788535,0.209327,ST_GNN_H32_BiasOn,GNN_and_GRU_BiasOn
3,2026-01-28 23:34:18,ST_GNN_H64_BiasOn,ST_GNN_biasinit,0.5,16,32,64,0.2,-2.9968,60,...,0.771457,0.005673,5018,4296,752910,26311,788535,0.211019,ST_GNN_H64_BiasOn,GNN_and_GRU_BiasOn


## experiments_log_gnn_biasinit_robust.csv

In [49]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids-mitre/logs_orig/experiments_log_gnn_biasinit_robust.csv")
df



,timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,...,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
0,2026-02-02 16:55:53,ST_GNN_H32_BiasOn,GNN_and_GRU_BiasOn,0.5,16,32,32,0.2,-2.9968,60,...,0.028464,0.224596,0.779844,0.00035,719,265,756941,30610,788535,0.225491


In [50]:
df["model_name_fix"] = ["ST_GNN_H32_BiasOn_robust"]
df["type_fix"] = ["ST_GNN_Robust"]

# Get current columns
current_columns = df.columns.tolist()

# Define columns to move
fixed_columns = ['model_name_fix', 'type_fix']
original_columns = ['model_name', 'type']

# Remove these columns from their current position
remaining_columns = [col for col in current_columns if col not in (['timestamp'] + fixed_columns + original_columns)]

# Construct the new order
new_column_order = [
    'timestamp',
    'model_name_fix',
    'type_fix'
] + remaining_columns + original_columns

# Apply the new order to the DataFrame
df = df[new_column_order]
df.to_csv("/content/drive/MyDrive/nids-mitre/logs/experiments_log_gnn_biasinit_robust.csv", index=False)

df

,timestamp,model_name_fix,type_fix,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,...,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss,model_name,type
0,2026-02-02 16:55:53,ST_GNN_H32_BiasOn_robust,ST_GNN_Robust,0.5,16,32,32,0.2,-2.9968,60,...,0.779844,0.00035,719,265,756941,30610,788535,0.225491,ST_GNN_H32_BiasOn,GNN_and_GRU_BiasOn


## experiments_log_gnn_biasinit_robust_identity.csv

In [75]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids-mitre/logs_orig/experiments_log_gnn_biasinit_robust_identity.csv")
df



,timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,...,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
0,2026-02-05 13:02:44,ST_GNN_H32_BiasOn_robust_identity,GNN_and_GRU_BiasOn,0.5,16,32,32,0.2,-2.9968,60,...,0.612108,0.726920,0.963963,0.006991,18220,5294,751912,13109,788535,0.091816
1,2026-02-05 14:16:12,Static_GNN_H32_BiasOn_robust_identity,GNN_static_BiasOn,0.5,16,32,32,0.2,-2.9968,60,...,0.357758,0.475986,0.894018,0.005251,9964,3976,753230,21365,788535,0.169509


In [76]:
df["type_fix"] = ["ST_GNN_Identity", "StaticGNN_Identity"]

# Get current columns
current_columns = df.columns.tolist()

# Define columns to move
fixed_columns = ['type_fix']
original_columns = ['type']

# Remove these columns from their current position
remaining_columns = [col for col in current_columns if col not in (['timestamp', 'model_name'] + fixed_columns + original_columns)]

# Construct the new order
new_column_order = [
    'timestamp',
    'model_name',
    'type_fix'
] + remaining_columns + original_columns

# Apply the new order to the DataFrame
df = df[new_column_order]
df.to_csv("/content/drive/MyDrive/nids-mitre/logs/experiments_log_gnn_biasinit_robust_identity.csv", index=False)

df

,timestamp,model_name,type_fix,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,...,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss,type
0,2026-02-05 13:02:44,ST_GNN_H32_BiasOn_robust_identity,ST_GNN_Identity,0.5,16,32,32,0.2,-2.9968,60,...,0.726920,0.963963,0.006991,18220,5294,751912,13109,788535,0.091816,GNN_and_GRU_BiasOn
1,2026-02-05 14:16:12,Static_GNN_H32_BiasOn_robust_identity,StaticGNN_Identity,0.5,16,32,32,0.2,-2.9968,60,...,0.475986,0.894018,0.005251,9964,3976,753230,21365,788535,0.169509,GNN_static_BiasOn


## experiments_log_mlp_biasinit.csv

In [77]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids-mitre/logs_orig/experiments_log_mlp_biasinit.csv")
df



,timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,...,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
0,2026-01-28 16:20:07,EdgeMLP,MLP,0.5,16,32,64,0.2,-2.996689,10,...,0.300437,0.084670,0.755455,0.336556,24303,254842,502364,7026,788535,0.939078
1,2026-01-28 17:06:42,EdgeMLP_W1_H32,MLP,0.5,16,32,32,0.2,-2.996800,10,...,0.000000,0.164224,0.762147,0.000000,0,0,757206,31329,788535,0.134474
2,2026-01-28 17:14:22,EdgeMLP_W1_H64,MLP,0.5,16,32,64,0.2,-2.996800,10,...,0.000000,0.157462,0.761873,0.000000,0,0,757206,31329,788535,0.133491
3,2026-01-28 17:21:37,EdgeMLP_W2_H32,MLP,0.5,16,32,32,0.2,-2.996800,10,...,0.000000,0.151047,0.754259,0.000000,0,0,757206,31329,788535,0.224974
4,2026-01-28 17:29:11,EdgeMLP_W2_H64,MLP,0.5,16,32,64,0.2,-2.996800,10,...,0.000000,0.155617,0.755854,0.000000,0,0,757206,31329,788535,0.223112
5,2026-01-28 17:36:27,EdgeMLP_W5_H32,MLP,0.5,16,32,32,0.2,-2.996800,10,...,0.000000,0.153098,0.757152,0.000000,0,0,757206,31329,788535,0.424834
6,2026-01-28 17:44:06,EdgeMLP_W5_H64,MLP,0.5,16,32,64,0.2,-2.996800,10,...,0.000000,0.155437,0.746614,0.000000,0,0,757206,31329,788535,0.423621


In [78]:
df["type_fix"] = ["EdgeMLP"]*len(df)

# Get current columns
current_columns = df.columns.tolist()

# Define columns to move
fixed_columns = ['type_fix']
original_columns = ['type']

# Remove these columns from their current position
remaining_columns = [col for col in current_columns if col not in (['timestamp', 'model_name'] + fixed_columns + original_columns)]

# Construct the new order
new_column_order = [
    'timestamp',
    'model_name',
    'type_fix'
] + remaining_columns + original_columns

# Apply the new order to the DataFrame
df = df[new_column_order]
df.to_csv("/content/drive/MyDrive/nids-mitre/logs/experiments_log_mlp_biasinit.csv", index=False)

df

,timestamp,model_name,type_fix,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,...,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss,type
0,2026-01-28 16:20:07,EdgeMLP,EdgeMLP,0.5,16,32,64,0.2,-2.996689,10,...,0.084670,0.755455,0.336556,24303,254842,502364,7026,788535,0.939078,MLP
1,2026-01-28 17:06:42,EdgeMLP_W1_H32,EdgeMLP,0.5,16,32,32,0.2,-2.996800,10,...,0.164224,0.762147,0.000000,0,0,757206,31329,788535,0.134474,MLP
2,2026-01-28 17:14:22,EdgeMLP_W1_H64,EdgeMLP,0.5,16,32,64,0.2,-2.996800,10,...,0.157462,0.761873,0.000000,0,0,757206,31329,788535,0.133491,MLP
3,2026-01-28 17:21:37,EdgeMLP_W2_H32,EdgeMLP,0.5,16,32,32,0.2,-2.996800,10,...,0.151047,0.754259,0.000000,0,0,757206,31329,788535,0.224974,MLP
4,2026-01-28 17:29:11,EdgeMLP_W2_H64,EdgeMLP,0.5,16,32,64,0.2,-2.996800,10,...,0.155617,0.755854,0.000000,0,0,757206,31329,788535,0.223112,MLP
5,2026-01-28 17:36:27,EdgeMLP_W5_H32,EdgeMLP,0.5,16,32,32,0.2,-2.996800,10,...,0.153098,0.757152,0.000000,0,0,757206,31329,788535,0.424834,MLP
6,2026-01-28 17:44:06,EdgeMLP_W5_H64,EdgeMLP,0.5,16,32,64,0.2,-2.996800,10,...,0.155437,0.746614,0.000000,0,0,757206,31329,788535,0.423621,MLP


## experiments_log_rf.csv

In [79]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids-mitre/logs_orig/experiments_log_rf.csv")
df

,timestamp,model_name,type,prob_threshold,hp_n_estimators,hp_class_weight,hp_max_depth,hp_n_jobs,hp_random_state,Precision,...,F1,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows
0,2026-01-28 16:26:51,RandomForest,Baseline_RF,0.5,100,balanced,15,-1,42,0.100119,...,0.179339,0.341435,0.286107,0.853607,0.319421,26915,241914,515438,4414,788681


In [80]:
df["type_fix"] = ["RandomForestClassifier"]

# Get current columns
current_columns = df.columns.tolist()

# Define columns to move
fixed_columns = ['type_fix']
original_columns = ['type']

# Remove these columns from their current position
remaining_columns = [col for col in current_columns if col not in (['timestamp', 'model_name'] + fixed_columns + original_columns)]

# Construct the new order
new_column_order = [
    'timestamp',
    'model_name',
    'type_fix'
] + remaining_columns + original_columns

# Apply the new order to the DataFrame
df = df[new_column_order]
df.to_csv("/content/drive/MyDrive/nids-mitre/logs/experiments_log_rf.csv", index=False)

df

,timestamp,model_name,type_fix,prob_threshold,hp_n_estimators,hp_class_weight,hp_max_depth,hp_n_jobs,hp_random_state,Precision,...,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,type
0,2026-01-28 16:26:51,RandomForest,RandomForestClassifier,0.5,100,balanced,15,-1,42,0.100119,...,0.341435,0.286107,0.853607,0.319421,26915,241914,515438,4414,788681,Baseline_RF


## experiments_log_rnn_biasinit.csv

In [90]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids-mitre/logs_orig/experiments_log_rnn_biasinit.csv")
df

,timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,...,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
0,2026-02-02 15:00:49,EdgeGRUrobust_BiasOn,GRU,0.5,16,32,32,0.2,-2.9968,60,...,0.328369,0.287438,0.873958,0.017032,9715,12897,744309,21614,788535,0.185354


In [91]:
df["type_fix"] = ["EdgeGRU_Baseline"]

# Get current columns
current_columns = df.columns.tolist()

# Define columns to move
fixed_columns = ['type_fix']
original_columns = ['type']

# Remove these columns from their current position
remaining_columns = [col for col in current_columns if col not in (['timestamp', 'model_name'] + fixed_columns + original_columns)]

# Construct the new order
new_column_order = [
    'timestamp',
    'model_name',
    'type_fix'
] + remaining_columns + original_columns

# Apply the new order to the DataFrame
df = df[new_column_order]
df.to_csv("/content/drive/MyDrive/nids-mitre/logs/experiments_log_rnn_biasinit_robust.csv", index=False)

df

,timestamp,model_name,type_fix,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,...,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss,type
0,2026-02-02 15:00:49,EdgeGRUrobust_BiasOn,EdgeGRU_Baseline,0.5,16,32,32,0.2,-2.9968,60,...,0.287438,0.873958,0.017032,9715,12897,744309,21614,788535,0.185354,GRU


## experiments_log_rnn_biasinitOLD.csv




In [65]:
!cat logs_orig/experiments_log_rnn_biasinitOLD.csv

timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_output_bias_init,extra_epochs,extra_batch_steps,extra_pos_weight,extra_learning_rate,Precision,Recall,F1,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
2026-02-02 12:35:50,EdgeGRU_BiasOn,GRU,0.5,16,32,32,-2.9968,60,10,2.0,0.005,0.0,0.0,0.0,0.0,0.05808773245201512,0.6578974912594538,0.0,0,0,757206,31329,788535,0.2532817155610178
2026-02-02 14:39:24,EdgeGRUrobust_BiasOn,GRU,0.5,16,32,32,0.2,-2.9968,60,10,2.0,0.005,0.42963912966566425,0.3100960771170481,0.3602083758180234,0.3283692066410686,0.2874379632310916,0.8739576550874,0.017032353150925904,9715,12897,744309,21614,788535,0.18535445860769845


In [68]:
!cat logs_orig/experiments_log_rnn_biasinit.csv

timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,extra_batch_steps,extra_pos_weight,extra_learning_rate,Precision,Recall,F1,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
2026-02-02 15:00:49,EdgeGRUrobust_BiasOn,GRU,0.5,16,32,32,0.2,-2.9968,60,10,2.0,0.005,0.42963912966566425,0.3100960771170481,0.3602083758180234,0.3283692066410686,0.2874379632310916,0.8739576550874,0.017032353150925904,9715,12897,744309,21614,788535,0.18535445860769845


Note: I can´t read logs_orig/experiments_log_rnn_biasinitOLD.csv with pandas, because the second experiment has an extra colums. That's why I created the file logs_orig/experiments_log_rnn_biasinit.csv

I will overwrite logs_orig/experiments_log_rnn_biasinitOLD.csv to mantain only the "old" experiment (in which I didn't use dropout)

In [84]:
%%writefile logs_orig/experiments_log_rnn_biasinitOLD.csv
timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_output_bias_init,extra_epochs,extra_batch_steps,extra_pos_weight,extra_learning_rate,Precision,Recall,F1,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
2026-02-02 12:35:50,EdgeGRU_BiasOn,GRU,0.5,16,32,32,-2.9968,60,10,2.0,0.005,0.0,0.0,0.0,0.0,0.05808773245201512,0.6578974912594538,0.0,0,0,757206,31329,788535,0.2532817155610178

Overwriting logs_orig/experiments_log_rnn_biasinitOLD.csv


In [92]:
!cat logs_orig/experiments_log_rnn_biasinitOLD.csv

timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_output_bias_init,extra_epochs,extra_batch_steps,extra_pos_weight,extra_learning_rate,Precision,Recall,F1,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
2026-02-02 12:35:50,EdgeGRU_BiasOn,GRU,0.5,16,32,32,-2.9968,60,10,2.0,0.005,0.0,0.0,0.0,0.0,0.05808773245201512,0.6578974912594538,0.0,0,0,757206,31329,788535,0.2532817155610178


In [93]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nids-mitre/logs_orig/experiments_log_rnn_biasinitOLD.csv")
df

,timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_output_bias_init,extra_epochs,extra_batch_steps,...,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
0,2026-02-02 12:35:50,EdgeGRU_BiasOn,GRU,0.5,16,32,32,-2.9968,60,10,...,0.0,0.058088,0.657897,0.0,0,0,757206,31329,788535,0.253282


In [94]:
df["type_fix"] = ["EdgeGRU_Baseline"]

# Get current columns
current_columns = df.columns.tolist()

# Define columns to move
fixed_columns = ['type_fix']
original_columns = ['type']

# Remove these columns from their current position
remaining_columns = [col for col in current_columns if col not in (['timestamp', 'model_name'] + fixed_columns + original_columns)]

# Construct the new order
new_column_order = [
    'timestamp',
    'model_name',
    'type_fix'
] + remaining_columns + original_columns

# Apply the new order to the DataFrame
df = df[new_column_order]
df.to_csv("/content/drive/MyDrive/nids-mitre/logs/experiments_log_rnn_biasinit.csv", index=False)

df

,timestamp,model_name,type_fix,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_output_bias_init,extra_epochs,extra_batch_steps,...,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss,type
0,2026-02-02 12:35:50,EdgeGRU_BiasOn,EdgeGRU_Baseline,0.5,16,32,32,-2.9968,60,10,...,0.058088,0.657897,0.0,0,0,757206,31329,788535,0.253282,GRU


In [95]:
!ls logs_orig/

experiments_log_gnn_biasinit.csv
experiments_log_gnn_biasinit_robust.csv
experiments_log_gnn_biasinit_robust_identity.csv
experiments_log_mlp_biasinit.csv
experiments_log_rf.csv
experiments_log_rnn_biasinit.csv
experiments_log_rnn_biasinitOLD.csv


In [97]:
!ls logs/

experiments_log_gnn_biasinit.csv
experiments_log_gnn_biasinit_robust.csv
experiments_log_gnn_biasinit_robust_identity.csv
experiments_log_mlp_biasinit.csv
experiments_log_rf.csv
experiments_log_rnn_biasinit.csv
experiments_log_rnn_biasinit_robust.csv


Note: `logs_orig/experiments_log_rnn_biasinit.csv` is the one where I used dropout, so I decided to change its name in the new folder: `logs/experiments_log_rnn_biasinit_robust.csv`

On the other hand, `logs_orig/experiments_log_rnn_biasinitOLD.csv` is the one where dropout was not defined, so I decided to change its name in the new folder: `logs/experiments_log_rnn_biasinit.csv`

In [100]:
!cat logs_orig/experiments_log_rnn_biasinit.csv
!echo "\n"
!cat logs/experiments_log_rnn_biasinit_robust.csv

timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,extra_batch_steps,extra_pos_weight,extra_learning_rate,Precision,Recall,F1,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
2026-02-02 15:00:49,EdgeGRUrobust_BiasOn,GRU,0.5,16,32,32,0.2,-2.9968,60,10,2.0,0.005,0.42963912966566425,0.3100960771170481,0.3602083758180234,0.3283692066410686,0.2874379632310916,0.8739576550874,0.017032353150925904,9715,12897,744309,21614,788535,0.18535445860769845
\n
timestamp,model_name,type_fix,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_dropout,hp_output_bias_init,extra_epochs,extra_batch_steps,extra_pos_weight,extra_learning_rate,Precision,Recall,F1,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss,type
2026-02-02 15:00:49,EdgeGRUrobust_BiasOn,EdgeGRU_Baseline,0.5,16,32,32,0.2,-2.9968,60,10,2.0,0.005,0.4296391296656642,0.3100960771170481,0.3602083758180234,0.3283692066410686,0.2874379632310916,0.8739576550874,0.01703235

In [101]:
!cat logs_orig/experiments_log_rnn_biasinitOLD.csv
!echo "\n"
!cat logs/experiments_log_rnn_biasinit.csv

timestamp,model_name,type,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_output_bias_init,extra_epochs,extra_batch_steps,extra_pos_weight,extra_learning_rate,Precision,Recall,F1,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss
2026-02-02 12:35:50,EdgeGRU_BiasOn,GRU,0.5,16,32,32,-2.9968,60,10,2.0,0.005,0.0,0.0,0.0,0.0,0.05808773245201512,0.6578974912594538,0.0,0,0,757206,31329,788535,0.2532817155610178
\n
timestamp,model_name,type_fix,prob_threshold,hp_node_dim,hp_edge_dim,hp_hidden_dim,hp_output_bias_init,extra_epochs,extra_batch_steps,extra_pos_weight,extra_learning_rate,Precision,Recall,F1,F2,AUC-PR,AUC-ROC,FPR,TP,FP,TN,FN,Total_Flows,Loss,type
2026-02-02 12:35:50,EdgeGRU_BiasOn,EdgeGRU_Baseline,0.5,16,32,32,-2.9968,60,10,2.0,0.005,0.0,0.0,0.0,0.0,0.0580877324520151,0.6578974912594538,0.0,0,0,757206,31329,788535,0.2532817155610178,GRU
